In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-miniLM-L6-v2")

sentences = [
    "Python generators produce values lazily.",
    "Generators allow values to be produced one at a time.",
    "I like eating chocolate cake."
]

embeddings = model.encode(sentences)

print("Number of sentences:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of sentences: 3
Vector dimensions: 384


Inspect the actual embedding

In [ ]:
embedding = embeddings[0]

print("Vector type:", type(embedding))
print("Vector length:", len(embedding))
print("First 20 values:")

print(embedding[:20])

Vector type: <class 'numpy.ndarray'>
Vector length: 384
First 20 values:
[-0.09992032  0.08044463 -0.00395372  0.10376896 -0.03657861 -0.10683947
  0.00328126 -0.03761544  0.00067721 -0.06610245  0.07842042 -0.01206757
  0.02601092 -0.1087542   0.02196711 -0.01347643 -0.026603   -0.04964894
 -0.04731206 -0.10735827]


Compare embeddings using cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)

print(similarity)

[[ 0.9999999   0.5902653  -0.06276894]
 [ 0.5902653   1.         -0.04643155]
 [-0.06276894 -0.04643155  0.99999976]]


In [ ]:
print("Sentence 1 vs Sentence 2:",
      similarity[0][1])

print("Sentence 1 vs Sentence 3:",
      similarity[0][2])

Sentence 1 vs Sentence 2: 0.5902653
Sentence 1 vs Sentence 3: -0.062768936


Semantic search

In [ ]:
documents = [
    "Python generators produce values lazily.",
    "FastAPI is a framework for building APIs.",
    "Machine learning models learn patterns from data.",
    "Python lists store collections of items.",
    "Chocolate cake is made with cocoa and flour."
]

In [ ]:
document_embeddings = model.encode(documents)

In [ ]:
def semantic_search(query, documents, document_embeddings, top_k=3):
  query_embedding = model.encode([query])

  scores = cosine_similarity(
      query_embedding,
      document_embeddings
  )[0]

  ranked_indices = scores.argsort()[::-1]

  results = []

  for index in ranked_indices[:top_k]:
    results.append({
        "document": documents[index],
        "score": scores[index]
    })

  return results

In [ ]:
query = "How does Python generate values one at a time?"

results = semantic_search(query, documents, document_embeddings)

for result in results:
  print(result["score"], result["document"])
  print()

0.67845887 Python generators produce values lazily.

0.41783616 Python lists store collections of items.

0.24889499 Machine learning models learn patterns from data.



In [ ]:
queries = [
    "How does Python lazily produce values?",
    "What framework can I use to create APIs in Python?",
    "How do machine learning systems learn?",
    "What is a dessert made using cocoa?"
]

for query in queries:
    print("=" * 80)
    print("QUERY:", query)

    results = semantic_search(
        query,
        documents,
        document_embeddings,
        top_k=2
    )

    for result in results:
        print(f"{result['score']:.4f} -> {result['document']}")

QUERY: How does Python lazily produce values?
0.7988 -> Python generators produce values lazily.
0.3987 -> Python lists store collections of items.
QUERY: What framework can I use to create APIs in Python?
0.6595 -> FastAPI is a framework for building APIs.
0.3154 -> Python lists store collections of items.
QUERY: How do machine learning systems learn?
0.7003 -> Machine learning models learn patterns from data.
0.0978 -> Python lists store collections of items.
QUERY: What is a dessert made using cocoa?
0.7293 -> Chocolate cake is made with cocoa and flour.
0.1081 -> FastAPI is a framework for building APIs.


In [ ]:
query = "What is a Python generator?"

results = semantic_search(
    query,
    documents,
    document_embeddings,
    top_k=5
)

for result in results:
    print(f"Score: {result['score']:.4f}")
    print(result["document"])
    print()

Score: 0.6409
Python generators produce values lazily.

Score: 0.3528
Python lists store collections of items.

Score: 0.2207
FastAPI is a framework for building APIs.

Score: 0.1817
Machine learning models learn patterns from data.

Score: -0.0230
Chocolate cake is made with cocoa and flour.



In [ ]:
query = "Tell me about making chocolate desserts."

results = semantic_search(
    query,
    documents,
    document_embeddings,
    top_k=5
)

for result in results:
    print(f"Score: {result['score']:.4f}")
    print(result["document"])
    print()

Score: 0.6274
Chocolate cake is made with cocoa and flour.

Score: 0.1116
Machine learning models learn patterns from data.

Score: 0.0715
FastAPI is a framework for building APIs.

Score: 0.0517
Python generators produce values lazily.

Score: 0.0358
Python lists store collections of items.



              INGESTION
                  │
Website ──→ Chunks ──→ Embeddings ──→ Vector DB
                                     
                                     
               QUERY
                  │
User ──→ Question ──→ Query Embedding
                           │
                           ↓
                     Vector DB search
                           │
                           ↓
                    Relevant chunks
                           │
                           ↓
                          LLM
                           │
                           ↓
                         Answer